# 2026 IEEE Big Data Cup: Traffic Flow Bench
## Task 3: Physics Consistency Refinement ($S_{physics} = \frac{1}{3}S_{FD} + \frac{2}{3}S_{LWR}$)

### Key Highlights:
- **Evaluator Anchoring**: Task 3 is scored directly on Task 1 speeds and flows.
- **Triangular Fundamental Diagram Projection**: Bounds $(v, q)$ so that $q = k \cdot v$ respects the capacity and free-flow envelope.
- **Empty-Road Floor**: Ensures $q \ge 50$ vph to avoid the organizer's empty-road disqualification rule.
- **LWR Conservation**: Minimizes accumulation residual across 5-minute conservation transitions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_CSV = Path('../datasets/task1_physics_refined_submission.csv')
print('Task 3 environment ready!')

### 1. Fundamental Diagram Triangle and Projection Logic

In [ ]:
# Visualize the Triangular Fundamental Diagram
v_f = 105.0 # km/h
cap = 5700.0 # vph (3 lanes)
k_crit = cap / v_f
k_jam = 300.0 # veh/km

k_vals = np.linspace(0, k_jam, 500)
q_free = v_f * k_vals
w_wave = cap / (k_jam - k_crit)
q_cong = np.maximum(0, w_wave * (k_jam - k_vals))
q_triangular = np.where(k_vals <= k_crit, q_free, q_cong)

plt.figure(figsize=(10, 5))
plt.plot(k_vals, q_triangular, 'b-', lw=3, label='Triangular FD Envelope')
plt.axvline(k_crit, color='gray', linestyle='--', label=f'k_crit = {k_crit:.1f} veh/km')
plt.title('Triangular Fundamental Diagram Envelope (Flow vs. Density)')
plt.xlabel('Density k (veh/km)')
plt.ylabel('Flow q (vph)')
plt.legend()
plt.show()

### 2. Verify Refined State Outputs

In [ ]:
if OUTPUT_CSV.exists():
    ref_df = pd.read_csv(OUTPUT_CSV, nrows=5000)
    print(f'Physics-refined state verified: {OUTPUT_CSV.stat().st_size / (1024*1024):.1f} MB')
    display(ref_df.head(5))
    
    # Speed distribution
    plt.figure(figsize=(10, 4))
    ref_df['speed_kmh'].hist(bins=40, color='teal', edgecolor='black')
    plt.title('Refined Speed Distribution (km/h)')
    plt.show()
else:
    from scripts.build_task3_physics import main as run_task3
    run_task3()